# Cricket Simulator Prototype

## AI USE DECLARATION
- Generative AI is used in this project to prototype ideas in Python code.
- The conceptualization has entirely been done by me without AI support.
- All the markdown text is written by me without AI support, and can be treated as a writing sample.

## Approach and Philosophy

This simulation project aims to create a plausible simulation of cricket matches. I aim to take advantage of discrete and clearly defined nature of scoring points (or 'runs') in cricket. As runs can only scored in response to discrete and well-defined events known as deliveries (analogous to 'pitches' in baseball), one can build a mathematically plausible model of a cricket match by simulating the results of individual deliveries (which I will conceptualise as a probability distribution), and adding the outcomes of the deliveries until a termination conditon is met for the innings.

### Brief overview of cricket

Cricket is a non-contact sport similar to baseball where the batters of a team accumulate runs by striking a ball delivered in a designated fashion ('deliveries') by the opposing team, whose players ('fielders') attempt to stop the batters from scoring those runs. Teams take turns to bat and field. In the simplest formats of the game ('limited overs cricket'), the team that bats first sets a 'target' score by striking balls and running on the pitch ('running between the wickets'). The team that bats second need to accumulate at least one more run than the number of runs accumulated by their opposing team to win. In this format the batting team must accumulate all their runs within a limited number of deliveries. On each delivery the fielding team has the chance to dismiss (or 'out') batters from the field, in which case a new undismissed batter must be sent by the batting team. If all but one of the batters have been dismissed ('all out'), the team stops accumulating scores and its turn to bat ('innings') ends.

### The outcome of a delivery as a probability distribution

A single delivery can result in a number of mutually exclusive outcomes:
- X runs scored by the batting team (X can be any positive number, but the most common outcomes are 1, 2, 3, 4, and 6).
- Batter dismissed
- Invalid delivery -- This means the fielding team has made a foul delivery, and must re-deliver the ball. Sometimes extra runs are awarded to the batting team ('extras', most commonly 1 extra run per invalid delivery).
- Dot ball -- no runs have been scored, and no batter has been dismissed on a valid delivery.


Thus, these outcomes can be conceptualise to arise from a probability distribution. The probability distribution for a specific delivery depends on a large number of factors that characterise a cricket game -- e.g. the skill level of the batters and fielders, the stage of the match, the environmental conditions of the day (e.g. it is harder to field when the ground is moist), emotional factors, etc. Once the rules governing these probability distribution are defined clearly, one can simulate individual deliveries and summing them up until any of the termination conditions are met for the innings, which can be defined as:
- All out (10 of 11 batters dismisssed)
- The required number of valid deliveries have been completed by the fielding team
- The (second) batting team has exceeded the target.

### Parametrizing each delivery

Each delivery must be assigned its own probability distribution for a realistic conception. The plausibility of the model can then be improved by making better estimations of the probability distribution of each delivery given the circumstances of the delivery, including the factors highlighted above. For example, a skilled batter must have higher probability of scoring more runs, while having less probability of being dismissed. A skilled bowler (designated person from the fielding team who delivers a ball) should have more probility of dismissing a batter and less probability of conceding runs. If a skilled batter faces a skilled bowler, the effects of both must be combined in a sensible way to get a plausible distribution.

I conceive of the following parameters, which must be interpreted sensibly as independent effects on the probability distribution:
- Batter skill level
- Bowler skill level
- Fielding team skill level
- Undismissed batters remaining
- Deliveries remaining
- Runs to reach target (for a team batting first, this can mean an imaginary mental target that teams can have)

The parametrization is the key to obtaining a plausible model with this approach. Future iterations of the model will involve more parameters.

In [4]:
"""
Cricket Simulator - A probabilistic simulation of cricket matches
"""

import random
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from enum import Enum
import json
from collections import Counter

class DeliveryOutcome(Enum):
    """Possible outcomes of a single delivery"""
    DOT = "dot"           # 0 runs, no dismissal
    ONE = "1"             # 1 run
    TWO = "2"             # 2 runs
    THREE = "3"           # 3 runs
    FOUR = "4"            # 4 runs (boundary)
    SIX = "6"             # 6 runs (boundary)
    WICKET = "wicket"     # Batter dismissed
    WIDE = "wide"         # Invalid delivery (extra run)
    NO_BALL = "no_ball"   # Invalid delivery (extra run)


@dataclass
class DeliveryProbabilities:
    """Probability distribution for a single delivery outcome"""
    dot: float
    one: float
    two: float
    three: float
    four: float
    six: float
    wicket: float
    wide: float
    no_ball: float
    
    def __post_init__(self):
        """Normalize probabilities to sum to 1"""
        total = sum([self.dot, self.one, self.two, self.three, 
                     self.four, self.six, self.wicket, self.wide, self.no_ball])
        if total > 0:
            self.dot /= total
            self.one /= total
            self.two /= total
            self.three /= total
            self.four /= total
            self.six /= total
            self.wicket /= total
            self.wide /= total
            self.no_ball /= total
    
    def get_outcome(self) -> DeliveryOutcome:
        """Sample a delivery outcome based on probabilities"""
        outcomes = [
            (DeliveryOutcome.DOT, self.dot),
            (DeliveryOutcome.ONE, self.one),
            (DeliveryOutcome.TWO, self.two),
            (DeliveryOutcome.THREE, self.three),
            (DeliveryOutcome.FOUR, self.four),
            (DeliveryOutcome.SIX, self.six),
            (DeliveryOutcome.WICKET, self.wicket),
            (DeliveryOutcome.WIDE, self.wide),
            (DeliveryOutcome.NO_BALL, self.no_ball)
        ]
        
        r = random.random()
        cumulative = 0
        for outcome, prob in outcomes:
            cumulative += prob
            if r <= cumulative:
                return outcome
        return DeliveryOutcome.DOT


@dataclass
class Player:
    """Represents a cricket player"""
    name: str
    batting_skill: float  # 0-100, higher is better
    bowling_skill: float  # 0-100, higher is better
    fielding_skill: float # 0-100, higher is better
    
    def __post_init__(self):
        # Clamp skills to valid range
        self.batting_skill = max(0, min(100, self.batting_skill))
        self.bowling_skill = max(0, min(100, self.bowling_skill))
        self.fielding_skill = max(0, min(100, self.fielding_skill))


@dataclass
class Team:
    """Represents a cricket team"""
    name: str
    players: List[Player]
    batting_order: List[int]  # Indices of players in batting order
    
    def __post_init__(self):
        self.current_batsmen = []  # Indices of current batsmen
        self.dismissed = []  # Indices of dismissed batsmen
        self.score = 0
        self.wickets = 0
        self.balls_faced = 0
        self.extras = 0
        self.batting_stats = {}  # Per-player stats
    
    def get_next_batsman(self) -> Optional[int]:
        """Get the next batsman in order"""
        for idx in self.batting_order:
            if idx not in self.dismissed and idx not in self.current_batsmen:
                return idx
        return None
    
    def initialize_innings(self):
        """Set up batting order for innings"""
        self.current_batsmen = []
        self.dismissed = []
        self.score = 0
        self.wickets = 0
        self.balls_faced = 0
        self.extras = 0
        self.batting_stats = {}
        
        # Set initial batsmen
        for _ in range(2):
            batsman = self.get_next_batsman()
            if batsman is not None:
                self.current_batsmen.append(batsman)
                self.batting_stats[batsman] = {
                    'runs': 0,
                    'balls': 0,
                    'fours': 0,
                    'sixes': 0,
                    'dismissed': False
                }


class MatchState(Enum):
    """Match states"""
    NOT_STARTED = "not_started"
    FIRST_INNINGS = "first_innings"
    SECOND_INNINGS = "second_innings"
    COMPLETED = "completed"


class CricketSimulator:
    """Main simulator class"""
    
    def __init__(self, team1: Team, team2: Team, max_overs: int = 20):
        """
        Initialize the simulator
        
        Args:
            team1: First team (bats first)
            team2: Second team (bats second)
            max_overs: Maximum overs per innings
        """
        self.team1 = team1
        self.team2 = team2
        self.max_overs = max_overs
        self.max_balls = max_overs * 6
        self.match_state = MatchState.NOT_STARTED
        self.innings = []
        self.winner = None
        self.match_summary = {}
        
    def calculate_delivery_probabilities(self, 
                                        batsman: Player,
                                        bowler: Player,
                                        team: Team,
                                        runs_to_target: int = None,
                                        balls_remaining: int = None,
                                        wickets_remaining: int = None) -> DeliveryProbabilities:
        """
        Calculate probability distribution for a delivery based on all factors
        
        This is the key parametrization function. It combines:
        - Batter skill
        - Bowler skill  
        - Fielding team skill
        - Match context (runs needed, balls remaining, wickets remaining)
        - Aggressiveness based on context
        
        Returns a DeliveryProbabilities object
        """
        
        # Base probabilities (default for average players in neutral conditions)
        base_probs = DeliveryProbabilities(
            dot=0.35,
            one=0.30,
            two=0.10,
            three=0.02,
            four=0.08,
            six=0.05,
            wicket=0.05,
            wide=0.03,
            no_ball=0.02
        )
        
        # Skill modifiers (normalized to 0-1 range)
        bat_skill = batsman.batting_skill / 100.0
        bowl_skill = bowler.bowling_skill / 100.0
        
        # Calculate net advantage (batter advantage - bowler advantage)
        # This represents the relative advantage of batter vs bowler
        advantage = (bat_skill - bowl_skill) * 0.5
        
        # Adjust probabilities based on skill advantage
        # Better batters score more, get out less
        # Better bowlers get more wickets, concede less
        
        if advantage > 0:  # Batter advantage
            # Increase scoring, decrease wicket
            base_probs.dot -= advantage * 0.15
            base_probs.one += advantage * 0.05
            base_probs.two += advantage * 0.02
            base_probs.four += advantage * 0.05
            base_probs.six += advantage * 0.03
            base_probs.wicket -= advantage * 0.10
        else:  # Bowler advantage
            # Decrease scoring, increase wicket
            base_probs.dot -= abs(advantage) * 0.10
            base_probs.one += abs(advantage) * 0.02
            base_probs.two += abs(advantage) * 0.01
            base_probs.four -= abs(advantage) * 0.03
            base_probs.six -= abs(advantage) * 0.02
            base_probs.wicket += abs(advantage) * 0.12
        
        # Context adjustments
        if runs_to_target is not None and balls_remaining is not None:
            # Calculate required run rate
            if balls_remaining > 0:
                required_rate = runs_to_target / balls_remaining
                
                # Higher required rate -> more aggressive batting
                if required_rate > 1.5:  # High required rate (in T20 context)
                    base_probs.dot -= 0.05
                    base_probs.one += 0.02
                    base_probs.two += 0.01
                    base_probs.four += 0.02
                    base_probs.six += 0.02
                    base_probs.wicket += 0.03  # More risk
                elif required_rate < 0.5:  # Low required rate
                    base_probs.dot += 0.05
                    base_probs.one += 0.02
                    base_probs.four -= 0.03
                    base_probs.six -= 0.02
                    base_probs.wicket -= 0.02
        
        # Wickets remaining adjustment
        if wickets_remaining is not None:
            if wickets_remaining <= 2:  # Tail-end batting
                base_probs.dot += 0.05
                base_probs.four -= 0.03
                base_probs.six -= 0.02
                base_probs.wicket += 0.03
            elif wickets_remaining >= 8:  # Top-order batting
                base_probs.four += 0.02
                base_probs.six += 0.01
                base_probs.wicket -= 0.02
        
        # Ensure no negative probabilities
        for attr in ['dot', 'one', 'two', 'three', 'four', 'six', 'wicket', 'wide', 'no_ball']:
            setattr(base_probs, attr, max(0, getattr(base_probs, attr)))
        
        return base_probs
    
    def simulate_delivery(self, 
                         batting_team: Team,
                         bowling_team: Team,
                         striker_idx: int,
                         bowler_idx: int,
                         runs_to_target: int = None,
                         balls_remaining: int = None) -> Tuple[DeliveryOutcome, int]:
        """
        Simulate a single delivery
        
        Returns:
            Tuple of (outcome, runs_scored_on_delivery)
        """
        batsman = batting_team.players[striker_idx]
        bowler = bowling_team.players[bowler_idx]
        
        # Get fielding skill (average of all fielders)
        fielding_skill = sum(p.fielding_skill for p in bowling_team.players) / len(bowling_team.players)
        
        # Calculate probabilities
        wickets_remaining = 11 - batting_team.wickets
        probs = self.calculate_delivery_probabilities(
            batsman, bowler, batting_team,
            runs_to_target, balls_remaining, wickets_remaining
        )
        
        # Sample outcome
        outcome = probs.get_outcome()
        
        # Calculate runs
        runs_scored = 0
        if outcome == DeliveryOutcome.DOT:
            runs_scored = 0
        elif outcome == DeliveryOutcome.ONE:
            runs_scored = 1
        elif outcome == DeliveryOutcome.TWO:
            runs_scored = 2
        elif outcome == DeliveryOutcome.THREE:
            runs_scored = 3
        elif outcome == DeliveryOutcome.FOUR:
            runs_scored = 4
        elif outcome == DeliveryOutcome.SIX:
            runs_scored = 6
        elif outcome == DeliveryOutcome.WIDE:
            runs_scored = 1  # Extra run
            batting_team.extras += 1
        elif outcome == DeliveryOutcome.NO_BALL:
            runs_scored = 1  # Extra run
            batting_team.extras += 1
        
        # Update stats
        if outcome != DeliveryOutcome.WIDE and outcome != DeliveryOutcome.NO_BALL:
            batting_team.balls_faced += 1
            if outcome != DeliveryOutcome.WICKET:
                batting_team.batting_stats[striker_idx]['runs'] += runs_scored
                batting_team.batting_stats[striker_idx]['balls'] += 1
                if outcome == DeliveryOutcome.FOUR:
                    batting_team.batting_stats[striker_idx]['fours'] += 1
                elif outcome == DeliveryOutcome.SIX:
                    batting_team.batting_stats[striker_idx]['sixes'] += 1
        
        # Handle wicket
        if outcome == DeliveryOutcome.WICKET:
            batting_team.wickets += 1
            batting_team.batting_stats[striker_idx]['dismissed'] = True
            batting_team.dismissed.append(striker_idx)
            
            # Replace batsman
            new_batsman = batting_team.get_next_batsman()
            if new_batsman is not None:
                # Replace striker with new batsman
                idx = batting_team.current_batsmen.index(striker_idx)
                batting_team.current_batsmen[idx] = new_batsman
                batting_team.batting_stats[new_batsman] = {
                    'runs': 0,
                    'balls': 0,
                    'fours': 0,
                    'sixes': 0,
                    'dismissed': False
                }
            else:
                # No more batsmen (all out)
                batting_team.current_batsmen.remove(striker_idx)
        
        # Update score
        if outcome != DeliveryOutcome.WICKET:
            batting_team.score += runs_scored
        
        # For invalid deliveries, no striker change (ball is re-bowled)
        # For valid deliveries, striker rotates on odd runs
        if outcome not in [DeliveryOutcome.WIDE, DeliveryOutcome.NO_BALL, DeliveryOutcome.WICKET]:
            if runs_scored % 2 == 1:
                batting_team.current_batsmen.reverse()
        
        return outcome, runs_scored
    
    def simulate_innings(self, batting_team: Team, bowling_team: Team, target: int = None) -> Dict:
        """
        Simulate a full innings
        
        Args:
            batting_team: Team batting
            bowling_team: Team bowling
            target: Target score (for second innings)
        
        Returns:
            Dict with innings summary
        """
        batting_team.initialize_innings()
        
        # Track innings stats
        balls_bowled = 0
        runs_scored = 0
        wickets_taken = 0
        
        # Track bowling stats
        bowling_stats = {i: {'balls': 0, 'runs': 0, 'wickets': 0} for i in range(len(bowling_team.players))}
        
        # Choose initial bowler
        bowler_idx = 0  # Simple rotation for now
        
        # Track overs
        overs_bowled = 0
        balls_in_over = 0
        
        # Determine if chasing target
        runs_to_target = target if target is not None else float('inf')
        
        while True:
            # Check termination conditions
            if batting_team.wickets >= 10:  # All out
                break
            
            if balls_bowled >= self.max_balls:
                break
            
            if target is not None and batting_team.score > target:
                break
            
            # Get current striker (first batsman in list)
            if not batting_team.current_batsmen:
                break  # No batsmen left
            
            striker_idx = batting_team.current_batsmen[0]
            
            # Simulate delivery
            balls_remaining = self.max_balls - balls_bowled - 1
            outcome, runs = self.simulate_delivery(
                batting_team, bowling_team, striker_idx, bowler_idx,
                runs_to_target - batting_team.score if target is not None else None,
                balls_remaining
            )
            
            # Update bowling stats for valid deliveries
            if outcome != DeliveryOutcome.WIDE and outcome != DeliveryOutcome.NO_BALL:
                balls_bowled += 1
                balls_in_over += 1
                bowling_stats[bowler_idx]['balls'] += 1
                if outcome == DeliveryOutcome.WICKET:
                    bowling_stats[bowler_idx]['wickets'] += 1
                if outcome != DeliveryOutcome.WICKET:
                    bowling_stats[bowler_idx]['runs'] += runs
            else:
                # Invalid delivery - runs still count but ball is re-bowled
                if outcome != DeliveryOutcome.WICKET:
                    bowling_stats[bowler_idx]['runs'] += runs
            
            # Change bowler after over
            if balls_in_over >= 6:
                overs_bowled += 1
                balls_in_over = 0
                bowler_idx = (bowler_idx + 1) % len(bowling_team.players)
        
        # Prepare innings summary
        innings_summary = {
            'team': batting_team.name,
            'score': batting_team.score,
            'wickets': batting_team.wickets,
            'overs': balls_bowled / 6.0,
            'balls_faced': balls_bowled,
            'extras': batting_team.extras,
            'batting_stats': batting_team.batting_stats,
            'bowling_stats': bowling_stats,
            'target_chased': target is not None,
            'achieved_target': target is not None and batting_team.score > target
        }
        
        return innings_summary
    
    def simulate_match(self) -> Dict:
        """
        Simulate a complete match
        """
        self.match_state = MatchState.FIRST_INNINGS
        
        # First innings
        innings1 = self.simulate_innings(self.team1, self.team2)
        self.innings.append(innings1)
        
        # Second innings
        self.match_state = MatchState.SECOND_INNINGS
        target = innings1['score']
        innings2 = self.simulate_innings(self.team2, self.team1, target)
        self.innings.append(innings2)
        
        self.match_state = MatchState.COMPLETED
        
        # Determine winner
        if innings2['achieved_target']:
            self.winner = self.team2.name
        elif innings2['score'] == target:
            self.winner = "Tie"
        else:
            self.winner = self.team1.name
        
        # Prepare match summary
        self.match_summary = {
            'team1': self.team1.name,
            'team2': self.team2.name,
            'innings1': innings1,
            'innings2': innings2,
            'winner': self.winner,
            'max_overs': self.max_overs
        }
        
        return self.match_summary
    
    def print_match_summary(self):
        """Print a formatted match summary"""
        if self.match_state != MatchState.COMPLETED:
            print("Match not yet completed")
            return
        
        print("\n" + "="*60)
        print(f"MATCH SUMMARY: {self.team1.name} vs {self.team2.name}")
        print("="*60)
        
        # Print innings 1
        in1 = self.innings[0]
        print(f"\n{self.team1.name} - {in1['score']}/{in1['wickets']} ({in1['overs']:.1f} overs)")
        print("Batting:")
        for idx, stats in in1['batting_stats'].items():
            player = self.team1.players[idx]
            status = "not out"
            if stats['dismissed']:
                status = "out"
            print(f"  {player.name}: {stats['runs']} ({stats['balls']} balls, {stats['fours']}x4, {stats['sixes']}x6) {status}")
        
        print("\nBowling:")
        for idx, stats in in1['bowling_stats'].items():
            if stats['balls'] > 0:
                player = self.team2.players[idx]
                overs = stats['balls'] / 6.0
                econ = stats['runs'] / (stats['balls'] / 6.0) if stats['balls'] > 0 else 0
                print(f"  {player.name}: {overs:.1f}-{stats['wickets']}-{stats['runs']} (Econ: {econ:.2f})")
        
        # Print innings 2
        in2 = self.innings[1]
        print(f"\n{self.team2.name} - {in2['score']}/{in2['wickets']} ({in2['overs']:.1f} overs)")
        print("Batting:")
        for idx, stats in in2['batting_stats'].items():
            player = self.team2.players[idx]
            status = "not out"
            if stats['dismissed']:
                status = "out"
            print(f"  {player.name}: {stats['runs']} ({stats['balls']} balls, {stats['fours']}x4, {stats['sixes']}x6) {status}")
        
        print("\nBowling:")
        for idx, stats in in2['bowling_stats'].items():
            if stats['balls'] > 0:
                player = self.team1.players[idx]
                overs = stats['balls'] / 6.0
                econ = stats['runs'] / (stats['balls'] / 6.0) if stats['balls'] > 0 else 0
                print(f"  {player.name}: {overs:.1f}-{stats['wickets']}-{stats['runs']} (Econ: {econ:.2f})")
        
        # Print result
        print("\n" + "="*60)
        print(f"RESULT: {self.winner} wins!")
        if self.winner != "Tie":
            margin = abs(in1['score'] - in2['score'])
            if in2['achieved_target']:
                print(f"  {self.team2.name} won by {10 - in2['wickets']} wickets")
            else:
                print(f"  {self.team1.name} won by {margin} runs")
        else:
            print("  Match tied!")
        print("="*60)


def create_sample_teams():
    """Create sample teams for testing"""
    # Create players
    players1 = [
        Player("Virat Kohli", 95, 30, 70),
        Player("Rohit Sharma", 90, 25, 65),
        Player("KL Rahul", 85, 20, 60),
        Player("Suryakumar Yadav", 88, 25, 65),
        Player("Hardik Pandya", 80, 75, 70),
        Player("Ravindra Jadeja", 75, 80, 85),
        Player("Jasprit Bumrah", 20, 90, 60),
        Player("Mohammed Shami", 15, 88, 55),
        Player("Yuzvendra Chahal", 10, 85, 50),
        Player("Bhuvneshwar Kumar", 25, 82, 65),
        Player("Axar Patel", 60, 78, 70)
    ]
    
    players2 = [
        Player("Jos Buttler", 90, 30, 75),
        Player("David Warner", 88, 25, 70),
        Player("Steve Smith", 85, 40, 65),
        Player("Glenn Maxwell", 87, 70, 70),
        Player("Mitchell Starc", 25, 88, 60),
        Player("Pat Cummins", 35, 85, 65),
        Player("Adam Zampa", 15, 82, 55),
        Player("Marcus Stoinis", 70, 75, 70),
        Player("Josh Hazlewood", 20, 87, 60),
        Player("Aaron Finch", 82, 20, 65),
        Player("Ashton Agar", 60, 78, 65)
    ]
    
    # Create teams with batting order
    team1 = Team(
        name="Team India",
        players=players1,
        batting_order=list(range(len(players1)))
    )
    
    team2 = Team(
        name="Team Australia",
        players=players2,
        batting_order=list(range(len(players2)))
    )
    
    return team1, team2


def run_simulation_example():
    """Run a sample simulation"""
    team1, team2 = create_sample_teams()
    
    simulator = CricketSimulator(team1, team2, max_overs=20)
    result = simulator.simulate_match()
    simulator.print_match_summary()
    
    return simulator, result



In [5]:

# Run the simulation
if __name__ == "__main__":
    # Run example simulation
    simulator, result = run_simulation_example()
    
    # Additional: Run multiple simulations to see statistics
    print("\n\nRunning multiple simulations for statistical analysis...")
    results = []
    for i in range(10):
        team1, team2 = create_sample_teams()
        sim = CricketSimulator(team1, team2, max_overs=20)
        result = sim.simulate_match()
        results.append(result['winner'])
    
    win_counts = Counter(results)
    print(f"\nWin distribution over 10 matches:")
    for team, wins in win_counts.items():
        print(f"  {team}: {wins} wins")


MATCH SUMMARY: Team India vs Team Australia

Team India - 166/2 (20.0 overs)
Batting:
  Virat Kohli: 87 (66 balls, 7x4, 4x6) not out
  Rohit Sharma: 62 (46 balls, 6x4, 2x6) out
  KL Rahul: 8 (4 balls, 0x4, 0x6) out
  Suryakumar Yadav: 0 (2 balls, 0x4, 0x6) not out

Bowling:
  Jos Buttler: 2.0-0-17 (Econ: 8.50)
  David Warner: 2.0-0-18 (Econ: 9.00)
  Steve Smith: 2.0-0-14 (Econ: 7.00)
  Glenn Maxwell: 2.0-0-15 (Econ: 7.50)
  Mitchell Starc: 2.0-0-27 (Econ: 13.50)
  Pat Cummins: 2.0-1-12 (Econ: 6.00)
  Adam Zampa: 2.0-0-6 (Econ: 3.00)
  Marcus Stoinis: 2.0-0-14 (Econ: 7.00)
  Josh Hazlewood: 2.0-1-17 (Econ: 8.50)
  Aaron Finch: 1.0-0-16 (Econ: 16.00)
  Ashton Agar: 1.0-0-10 (Econ: 10.00)

Team Australia - 171/2 (17.7 overs)
Batting:
  Jos Buttler: 93 (51 balls, 7x4, 5x6) out
  David Warner: 53 (40 balls, 4x4, 3x6) out
  Steve Smith: 14 (11 balls, 2x4, 0x6) not out
  Glenn Maxwell: 7 (2 balls, 0x4, 1x6) not out

Bowling:
  Virat Kohli: 2.0-0-26 (Econ: 13.00)
  Rohit Sharma: 2.0-0-19 (Eco